# 🔨 Taller: Construye tu propio Pipeline RAG

**Curso:** Calidad de Software y Pruebas Automatizadas — Universidad de Nariño

**Módulo 1:** Requirements Refiner

---

## Objetivo

Al finalizar este taller, cada grupo habrá construido un **sistema RAG funcional** para su propio proyecto. No vas a mirar código ajeno — vas a construir el tuyo.

## Instrucciones

1. Formen grupos de 2-3 personas
2. Elijan un proyecto (real o académico): sistema de biblioteca, app de delivery, sistema de notas, e-commerce, etc.
3. En cada sección van a escribir código y responder preguntas
4. Al final, cada grupo presenta sus resultados al curso

## Estructura del taller

| Sección | Concepto | Qué van a construir | Tiempo |
|---------|----------|---------------------|--------|
| 1 | Tokenización | Analizar cómo el modelo ve los textos de su proyecto | 15 min |
| 2 | Embeddings | Convertir sus requerimientos en vectores | 15 min |
| 3 | Similitud Coseno | Descubrir qué requerimientos se parecen | 20 min |
| 4 | ChromaDB | Construir su base de conocimiento | 25 min |
| 5 | RAG + LLM | Generar historias de usuario con su KB | 20 min |
| Final | Presentación | Mostrar resultados al curso | 15 min |

**Tiempo total estimado: 2 horas**

---
## 📦 Paso 0: Instalación (ejecutar una sola vez)

In [ ]:
!pip install -q sentence-transformers chromadb groq
print("✅ Dependencias instaladas")

---
## 📋 Paso 0.5: Definan su proyecto

Antes de empezar, completen la información de su grupo.

In [ ]:
# ============================================================
# ✏️ COMPLETEN CON LOS DATOS DE SU GRUPO
# ============================================================

NOMBRE_GRUPO = ""           # Ejemplo: "Grupo Alpha"
INTEGRANTES = ""            # Ejemplo: "Ana, Carlos, María"
NOMBRE_PROYECTO = ""        # Ejemplo: "Sistema de Gestión de Biblioteca"
DESCRIPCION_PROYECTO = ""   # Ejemplo: "Aplicación web para gestionar préstamos de libros..."

# Verificación
assert NOMBRE_GRUPO, "⚠️ Escriban el nombre de su grupo"
assert NOMBRE_PROYECTO, "⚠️ Escriban el nombre de su proyecto"
print(f"✅ Grupo: {NOMBRE_GRUPO}")
print(f"   Proyecto: {NOMBRE_PROYECTO}")
print(f"   Integrantes: {INTEGRANTES}")
print(f"   Descripción: {DESCRIPCION_PROYECTO}")

---
## Sección 1: Tokenización (15 min)

### Concepto

Un LLM no lee texto como nosotros. Descompone cada frase en **tokens**: fragmentos que pueden ser palabras completas, partes de palabras o caracteres. El tokenizador es específico de cada modelo — fue entrenado junto con él.

### ¿Por qué importa?

Si el tokenizador fragmenta demasiado una palabra (porque no la conoce), el modelo arranca con desventaja para entender su significado. Un modelo entrenado con poco español fragmentará más las palabras en español.

### 🔨 Su tarea

Escriban **5 requerimientos de su proyecto** y observen cómo los tokeniza el modelo.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# ============================================================
# ✏️ ESCRIBAN 5 REQUERIMIENTOS DE SU PROYECTO
# Deben ser requerimientos reales, no inventados al azar.
# Mezclen algunos similares y otros diferentes.
# ============================================================

mis_requerimientos = [
    "",  # Requerimiento 1
    "",  # Requerimiento 2
    "",  # Requerimiento 3
    "",  # Requerimiento 4
    "",  # Requerimiento 5
]

# Verificar que escribieron los requerimientos
assert all(r.strip() for r in mis_requerimientos), "⚠️ Completen todos los requerimientos"

print(f"Proyecto: {NOMBRE_PROYECTO}")
print("=" * 60)

for i, req in enumerate(mis_requerimientos, 1):
    tokens = tokenizer.tokenize(req)
    print(f"\n📝 Req {i}: \"{req}\"")
    print(f"   Tokens ({len(tokens)}): {tokens}")

### 📝 Entregable Sección 1

Respondan estas preguntas en la celda de abajo:

In [ ]:
# ============================================================
# ✏️ RESPONDAN AQUÍ (escriban dentro de las comillas triples)
# ============================================================

respuesta_1_1 = """
¿Cuál requerimiento generó más tokens y por qué creen que pasó?
Respuesta: 
"""

respuesta_1_2 = """
¿Encontraron alguna palabra que el modelo fragmentó de forma inesperada? ¿Cuál?
Respuesta: 
"""

respuesta_1_3 = """
Si este modelo fue entrenado principalmente en inglés, ¿qué implicaciones
tiene para un proyecto donde los requerimientos se escriben en español?
Respuesta: 
"""

print("✅ Respuestas guardadas")

---
## Sección 2: Embeddings (15 min)

### Concepto

Un **embedding** es un vector (lista de números) que captura el **significado semántico** de un texto. Cada texto se convierte en un punto en un espacio de 384 dimensiones. Textos con significado similar quedan ubicados cerca.

### Diferencia con los tokens
- Tokens = fragmentos sin contexto (piezas sueltas)
- Embedding = representación del significado **completo** de la oración

### 🔨 Su tarea

Generen los embeddings de sus requerimientos y observen las propiedades del vector.

In [ ]:
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

print("⏳ Cargando modelo de embeddings...")
modelo = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Modelo cargado")

# Generar embeddings de SUS requerimientos
embeddings = modelo.encode(mis_requerimientos)

print(f"\n📊 Resultados:")
print(f"   Cantidad de textos: {len(mis_requerimientos)}")
print(f"   Dimensiones por embedding: {embeddings.shape[1]}")
print(f"   Forma total: {embeddings.shape}")

for i, req in enumerate(mis_requerimientos):
    print(f"\n📝 Req {i+1}: \"{req[:60]}...\"")
    print(f"   Primeros 5 valores: [{', '.join(f'{v:.4f}' for v in embeddings[i][:5])}  ...]")
    print(f"   Magnitud del vector: {sum(v**2 for v in embeddings[i])**0.5:.4f}")

### 📝 Entregable Sección 2

Respondan estas preguntas:

In [ ]:
# ============================================================
# ✏️ RESPONDAN AQUÍ
# ============================================================

respuesta_2_1 = """
¿Los 384 números del embedding significan algo individualmente?
¿O solo tienen sentido como conjunto? Expliquen.
Respuesta: 
"""

respuesta_2_2 = """
¿Los embeddings de sus requerimientos se generaron comparándose entre sí,
o cada uno se generó de forma independiente? ¿Por qué?
Respuesta: 
"""

print("✅ Respuestas guardadas")

---
## Sección 3: Similitud Coseno (20 min)

### Concepto

La **similitud coseno** mide el ángulo entre dos vectores:
- **1.0** = misma dirección → significado idéntico
- **0.0** = perpendiculares → sin relación

Usamos coseno y no distancia euclidiana porque el coseno solo mide **dirección** (significado), no magnitud (longitud del texto).

### Umbrales prácticos
- **> 0.7** = muy similares
- **0.4 - 0.7** = algo relacionados
- **< 0.4** = no relacionados

### 🔨 Su tarea

Calculen la matriz de similitud entre sus requerimientos y analicen los resultados.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similitudes = cosine_similarity(embeddings)

print(f"Proyecto: {NOMBRE_PROYECTO}")
print("=" * 60)
print("MATRIZ DE SIMILITUD COSENO")
print("=" * 60)

# Mostrar todos los pares
pares = []
for i in range(len(mis_requerimientos)):
    for j in range(i + 1, len(mis_requerimientos)):
        val = similitudes[i][j]
        emoji = "🟢" if val > 0.7 else "🟡" if val > 0.4 else "🔴"
        pares.append((val, i, j, emoji))
        print(f"{emoji} {val:.3f}  Req {i+1} ↔ Req {j+1}")
        print(f"         \"{mis_requerimientos[i][:50]}...\"")
        print(f"         \"{mis_requerimientos[j][:50]}...\"")
        print()

# Resumen
pares.sort(reverse=True)
print("\n📊 RANKING DE SIMILITUD:")
print(f"   🥇 Más similares: Req {pares[0][1]+1} ↔ Req {pares[0][2]+1} ({pares[0][0]:.3f})")
print(f"   🥉 Menos similares: Req {pares[-1][1]+1} ↔ Req {pares[-1][2]+1} ({pares[-1][0]:.3f})")

### 📝 Entregable Sección 3

Esta es la sección más importante para el pensamiento crítico.

In [ ]:
# ============================================================
# ✏️ RESPONDAN AQUÍ
# ============================================================

respuesta_3_1 = """
¿Cuáles requerimientos el modelo considera más similares?
¿Están de acuerdo con ese resultado? ¿Por qué sí o por qué no?
Respuesta: 
"""

respuesta_3_2 = """
¿Hay algún par de requerimientos que ustedes consideran similares
pero el modelo les da un puntaje bajo? ¿Por qué creen que pasa?
Respuesta: 
"""

respuesta_3_3 = """
Si usaran una búsqueda por texto (LIKE '%palabra%' en SQL)
en vez de similitud coseno, ¿encontrarían los mismos pares
similares? ¿Qué se perderían?
Respuesta: 
"""

print("✅ Respuestas guardadas")

---
## Sección 4: ChromaDB — Construyan su Base de Conocimiento (25 min)

### Concepto

ChromaDB es una **base de datos vectorial**: almacena embeddings y permite buscar los más cercanos a una consulta. Es como Google pero para significado — no busca por palabras clave sino por cercanía semántica.

### 🔨 Su tarea (la más importante del taller)

Van a escribir **5 historias de usuario de su proyecto** con criterios de aceptación Given/When/Then y cargarlas en ChromaDB. Estas historias serán la base de conocimiento que el RAG usará para generar nuevas historias.

**No tienen que ser perfectas** — justamente vamos a ver cómo el RAG las usa como referencia.

In [ ]:
# ============================================================
# ✏️ ESCRIBAN 5 HISTORIAS DE USUARIO DE SU PROYECTO
#
# Formato de cada historia:
#   - id: identificador único (ej: "MI-US-001")
#   - texto: "Como [rol], quiero [acción], para que [beneficio]"
#   - criterios: "GIVEN [precondición] WHEN [acción] THEN [resultado]"
#   - dominio: categoría temática (ej: "autenticacion", "inventario")
#
# IMPORTANTE: Piensen en historias de diferentes partes de su
# sistema. Si todas son del mismo módulo, el RAG no podrá
# diferenciar bien.
# ============================================================

mis_historias = [
    {
        "id": "MI-US-001",
        "texto": "",  # ✏️ Escriban aquí la historia 1
        "criterios": "",  # ✏️ Criterios Given/When/Then
        "dominio": "",  # ✏️ Dominio (ej: "autenticacion", "reportes")
    },
    {
        "id": "MI-US-002",
        "texto": "",
        "criterios": "",
        "dominio": "",
    },
    {
        "id": "MI-US-003",
        "texto": "",
        "criterios": "",
        "dominio": "",
    },
    {
        "id": "MI-US-004",
        "texto": "",
        "criterios": "",
        "dominio": "",
    },
    {
        "id": "MI-US-005",
        "texto": "",
        "criterios": "",
        "dominio": "",
    },
]

# Verificar que completaron todo
for i, h in enumerate(mis_historias, 1):
    assert h["texto"].strip(), f"⚠️ Falta el texto de la historia {i}"
    assert h["criterios"].strip(), f"⚠️ Faltan los criterios de la historia {i}"
    assert h["dominio"].strip(), f"⚠️ Falta el dominio de la historia {i}"

print(f"✅ {len(mis_historias)} historias definidas para el proyecto '{NOMBRE_PROYECTO}'")
for h in mis_historias:
    print(f"   [{h['id']}] ({h['dominio']}) {h['texto'][:80]}...")

In [ ]:
import chromadb

# Crear base de datos en memoria (para el taller)
client_db = chromadb.Client()
mi_collection = client_db.create_collection(
    name="mi_proyecto",
    metadata={"hnsw:space": "cosine"},  # Usar similitud coseno
)

# Generar embeddings de las historias
textos_historias = [h["texto"] for h in mis_historias]
embeddings_historias = modelo.encode(textos_historias).tolist()

# Indexar en ChromaDB
mi_collection.add(
    ids=[h["id"] for h in mis_historias],
    embeddings=embeddings_historias,
    documents=textos_historias,
    metadatas=[{
        "dominio": h["dominio"],
        "criterios": h["criterios"],
    } for h in mis_historias],
)

print(f"✅ {mi_collection.count()} historias indexadas en ChromaDB")
print(f"   Cada historia almacenada con: embedding ({embeddings.shape[1]} dim) + texto + metadatos")

### 🔍 Prueben su base de conocimiento

Ahora escriban un requerimiento NUEVO (que no esté en las historias) y busquen en su KB.

In [ ]:
# ============================================================
# ✏️ ESCRIBAN UN REQUERIMIENTO NUEVO PARA BUSCAR
# Debe ser diferente a las historias que cargaron pero
# relacionado con alguna de ellas.
# ============================================================

mi_busqueda = ""  # ✏️ Escriban aquí

assert mi_busqueda.strip(), "⚠️ Escriban un requerimiento para buscar"

# Buscar en ChromaDB
query_emb = modelo.encode([mi_busqueda]).tolist()
resultados = mi_collection.query(
    query_embeddings=query_emb,
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(f"🔍 Búsqueda: \"{mi_busqueda}\"")
print("=" * 60)

for i in range(len(resultados["ids"][0])):
    sim = 1 - resultados["distances"][0][i]
    emoji = "🟢" if sim > 0.5 else "🟡" if sim > 0.3 else "🔴"
    print(f"\n{emoji} #{i+1} [{resultados['ids'][0][i]}] Similitud: {sim:.3f}")
    print(f"   Dominio: {resultados['metadatas'][0][i]['dominio']}")
    print(f"   Historia: {resultados['documents'][0][i][:120]}...")

### 📝 Entregable Sección 4

In [ ]:
# ============================================================
# ✏️ RESPONDAN AQUÍ
# ============================================================

respuesta_4_1 = """
¿ChromaDB encontró la historia correcta para su búsqueda?
¿El ranking (orden de resultados) tiene sentido?
Respuesta: 
"""

respuesta_4_2 = """
¿Podrían haber encontrado la misma historia con una búsqueda
SQL tipo LIKE? ¿Qué ventaja tiene la búsqueda semántica?
Respuesta: 
"""

respuesta_4_3 = """
Si su base de conocimiento tuviera 1000 historias en vez de 5,
¿cambiaría algo en cómo funciona la búsqueda? ¿Sería más lenta?
Respuesta: 
"""

print("✅ Respuestas guardadas")

---
## Sección 5: RAG + LLM — El Pipeline Completo (20 min)

### Concepto

**RAG (Retrieval-Augmented Generation)** conecta todo:

1. Llega un requerimiento nuevo
2. Se genera su embedding (Sección 2)
3. Se buscan historias similares en ChromaDB (Sección 4)
4. Las historias encontradas se inyectan en el prompt del LLM
5. El LLM genera con **contexto institucional**, no genéricamente

### 🔨 Su tarea

Van a ver su pipeline completo funcionando: el LLM va a generar una historia de usuario usando SU base de conocimiento como referencia.

In [ ]:
from groq import Groq

# ============================================================
# ✏️ PEGUEN SU API KEY DE GROQ
# La obtienen gratis en: https://console.groq.com
# ============================================================

GROQ_API_KEY = ""  # ✏️ Peguen su API key aquí

assert GROQ_API_KEY.strip(), "⚠️ Necesitan una API key de Groq"
client_groq = Groq(api_key=GROQ_API_KEY)
print("✅ Groq conectado")

In [ ]:
def mi_pipeline_rag(requerimiento, collection, modelo, client_groq):
    """Pipeline RAG completo: Requisito → ChromaDB → Prompt → LLM → Historia"""

    # PASO 1: Buscar historias similares
    print("\n🔍 Paso 1: Buscando historias similares en su KB...")
    query_emb = modelo.encode([requerimiento]).tolist()
    resultados = collection.query(
        query_embeddings=query_emb,
        n_results=3,
        include=["documents", "metadatas", "distances"],
    )

    for i in range(len(resultados["ids"][0])):
        sim = 1 - resultados["distances"][0][i]
        print(f"   [{resultados['ids'][0][i]}] Similitud: {sim:.3f}")

    # PASO 2: Construir contexto RAG
    print("\n📦 Paso 2: Construyendo prompt enriquecido...")
    contexto = "## HISTORIAS DE REFERENCIA DE SU PROYECTO\n"
    contexto += "Usa estas historias como modelo de calidad:\n\n"

    for i in range(len(resultados["ids"][0])):
        sim = 1 - resultados["distances"][0][i]
        contexto += f"### Referencia {i+1} [{resultados['ids'][0][i]}] (similitud: {sim:.2f})\n"
        contexto += f"**Historia:** {resultados['documents'][0][i]}\n"
        contexto += f"**Criterios:** {resultados['metadatas'][0][i].get('criterios', '')}\n\n"

    system_prompt = f"""Eres un Analista de Requerimientos Senior.
Transforma requerimientos ambiguos en historias de usuario profesionales
con criterios de aceptación Given/When/Then verificables.

REGLAS:
1. Formato: "Como [rol], quiero [acción], para que [beneficio]"
2. Criterios Given/When/Then con validaciones específicas
3. Incluir validaciones de datos y tiempos de respuesta
4. Incluir manejo de errores y casos límite
5. Usar el mismo nivel de detalle que las historias de referencia

{contexto}"""

    user_message = f"""Transforma este requerimiento en historias de usuario
con el mismo nivel de calidad que las historias de referencia.

REQUERIMIENTO: {requerimiento}"""

    print(f"   Prompt total: {len(system_prompt) + len(user_message)} caracteres")

    # PASO 3: Generar con Groq
    print("\n🤖 Paso 3: Generando con Groq...")
    respuesta = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        temperature=0.3,
        max_tokens=2000,
    )

    resultado = respuesta.choices[0].message.content
    print(f"   ✅ Generado ({respuesta.usage.total_tokens} tokens)")

    return resultado

print("✅ Función del pipeline lista")

In [ ]:
# ============================================================
# ✏️ ESCRIBAN UN REQUERIMIENTO NUEVO PARA SU PROYECTO
# Debe ser algo que NO esté en las 5 historias que cargaron,
# pero que esté relacionado con su proyecto.
# ============================================================

mi_requerimiento_nuevo = ""  # ✏️ Escriban aquí

assert mi_requerimiento_nuevo.strip(), "⚠️ Escriban un requerimiento"

print(f"📝 Requerimiento: \"{mi_requerimiento_nuevo}\"")

# ============================================================
# GENERAR CON RAG (usa su base de conocimiento)
# ============================================================
print("\n" + "=" * 60)
print("✅ GENERACIÓN CON RAG (con su base de conocimiento)")
print("=" * 60)
resultado_con_rag = mi_pipeline_rag(
    mi_requerimiento_nuevo, mi_collection, modelo, client_groq
)
print("\n" + resultado_con_rag)

In [ ]:
# ============================================================
# GENERAR SIN RAG (para comparar)
# ============================================================
print("=" * 60)
print("❌ GENERACIÓN SIN RAG (genérica, sin su KB)")
print("=" * 60)

respuesta_sin = client_groq.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "Eres un Analista de Requerimientos. Responde en español."},
        {"role": "user", "content": f"Transforma este requerimiento en historias de usuario con criterios Given/When/Then.\n\nREQUERIMIENTO: {mi_requerimiento_nuevo}"},
    ],
    temperature=0.3,
    max_tokens=2000,
)

resultado_sin_rag = respuesta_sin.choices[0].message.content
print(resultado_sin_rag)

# Resumen comparativo
print("\n\n" + "=" * 60)
print("📊 COMPARACIÓN")
print("=" * 60)
print(f"Sin RAG: {len(resultado_sin_rag)} caracteres")
print(f"Con RAG: {len(resultado_con_rag)} caracteres")
print(f"\n💡 Comparen el nivel de detalle en los criterios Given/When/Then.")
print(f"   ¿Cuál se parece más al estilo de las historias que escribieron?")

### 📝 Entregable Sección 5 (Final)

In [ ]:
# ============================================================
# ✏️ RESPONDAN AQUÍ
# ============================================================

respuesta_5_1 = """
¿Qué diferencias concretas encontraron entre la generación
con RAG y sin RAG? Mencionen al menos 3 diferencias específicas.
Respuesta: 
"""

respuesta_5_2 = """
¿La historia generada con RAG se parece en estilo y nivel de
detalle a las historias que escribieron en su KB? ¿En qué sí y en qué no?
Respuesta: 
"""

respuesta_5_3 = """
Si tuvieran 100 historias en vez de 5 en su base de conocimiento,
¿cómo creen que cambiaría la calidad del resultado? ¿Por qué?
Respuesta: 
"""

respuesta_5_4 = """
¿En qué casos reales de su vida profesional o académica
les sería útil un sistema como este?
Respuesta: 
"""

print("✅ Respuestas guardadas")

---
## 🎯 Resumen: Lo que construyeron hoy

Cada grupo construyó un **pipeline RAG funcional** con 5 componentes:

```
Requerimiento nuevo
       │
       ▼
  [Tokenización] → El modelo descompone el texto en fragmentos
       │
       ▼
  [Embedding] → Convierte el texto en un vector de 384 números
       │
       ▼
  [ChromaDB] → Busca las historias más similares en la KB
       │
       ▼
  [Prompt RAG] → Inyecta las historias encontradas en el prompt
       │
       ▼
  [LLM (Groq)] → Genera historias de usuario con contexto
       │
       ▼
  Historia de usuario de alta calidad
```

### Para la presentación al curso

Cada grupo debe presentar (3-5 minutos):

1. **Su proyecto** — qué sistema eligieron
2. **Un hallazgo de la Sección 3** — algo que les sorprendió de la similitud coseno
3. **La comparación Sin RAG vs Con RAG** — muestren las dos salidas y expliquen las diferencias
4. **Una reflexión** — ¿cómo usarían esto en un proyecto real?

In [ ]:
# ============================================================
# RESUMEN DEL GRUPO (se genera automáticamente)
# ============================================================

print("=" * 60)
print(f"RESUMEN — {NOMBRE_GRUPO}")
print(f"Proyecto: {NOMBRE_PROYECTO}")
print(f"Integrantes: {INTEGRANTES}")
print("=" * 60)
print(f"\n📊 Estadísticas del taller:")
print(f"   Requerimientos analizados: {len(mis_requerimientos)}")
print(f"   Historias en la KB: {mi_collection.count()}")
print(f"   Modelo de embeddings: all-MiniLM-L6-v2 ({embeddings.shape[1]} dimensiones)")
print(f"   LLM utilizado: Llama 3.1 8B (via Groq)")
print(f"\n✅ Pipeline RAG funcionando para el proyecto '{NOMBRE_PROYECTO}'")